In [1]:
import soccerdata as sd
import pandas as pd
import time

[05/31/26 17:49:04] INFO     No custom team name replacements found. You can configure these in       _config.py:92
                             C:\Users\vibha\soccerdata\config\teamname_replacements.json.                          

                    INFO     No custom league dict found. You can configure additional leagues in    _config.py:190
                             C:\Users\vibha\soccerdata\config\league_dict.json.                                    

In [2]:
top_5_leagues = [
    "ENG-Premier League", 
    "ESP-La Liga", 
    "ITA-Serie A", 
    "GER-Bundesliga", 
    "FRA-Ligue 1"
]

fbref = sd.FBref(leagues=top_5_leagues, seasons=2025)
stat_categories = ["standard", "shooting", "playing_time", "misc","keeper"]

                    INFO     Saving cached data to C:\Users\vibha\soccerdata\data\FBref              _common.py:250

[05/31/26 17:49:13] WARNING  d:\Football Project\venv\Lib\site-packages\soccerdata\fbref.py:104:    ]8;id=12942768;file://C:\Python313\Lib\warnings.py\warnings.py]8;;\:]8;id=12942769;file://C:\Python313\Lib\warnings.py#110\110]8;;\
                             UserWarning: You are trying to scrape data for all of the Big 5                       
                             European leagues. This can be done more efficiently by setting                        
                             leagues='Big 5 European Leagues Combined'.                                            
                               warnings.warn(                                                                      
                                                                                                                   

In [3]:
master_df = fbref.read_player_season_stats(stat_type=stat_categories[0])
time.sleep(3)

In [4]:
for category in stat_categories[1:]:
    print(f"Fetching {category} stats...")
    cat_df = fbref.read_player_season_stats(stat_type=category)
    
    master_df = master_df.merge(
        cat_df,
        left_index=True,
        right_index=True,
        how='left',
        suffixes=('', '_drop')
    )
    
    master_df = master_df.filter(regex='^(?!.*_drop)')
    time.sleep(3)

Fetching shooting stats...
Fetching playing_time stats...
Fetching misc stats...
Fetching keeper stats...


In [5]:
master_df = master_df.reset_index()
master_df.columns = [
    "_".join(col).strip("_") if isinstance(col, tuple) else col
    for col in master_df.columns
]

In [6]:
master_df.head(50)

,league,season,team,player,nation,pos,age,born,Playing Time_MP,Playing Time_Starts,...,Performance_W,Performance_D,Performance_L,Performance_CS,Performance_CS%,Penalty Kicks_PKatt,Penalty Kicks_PKA,Penalty Kicks_PKsv,Penalty Kicks_PKm,Penalty Kicks_Save%
0,ENG-Premier League,2526,Arsenal,Ben White,ENG,DF,27,1997,12,9,...,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>
1,ENG-Premier League,2526,Arsenal,Bukayo Saka,ENG,"FW,MF",23,2001,31,25,...,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>
2,ENG-Premier League,2526,Arsenal,Christian Nørgaard,DEN,MF,31,1994,7,1,...,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>
3,ENG-Premier League,2526,Arsenal,Cristhian Mosquera,ESP,DF,21,2004,20,9,...,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>
4,ENG-Premier League,2526,Arsenal,David Raya,ESP,GK,29,1995,37,37,...,25,7,5,19,51.4,0,0,0,0,<NA>
5,ENG-Premier League,2526,Arsenal,Declan Rice,ENG,MF,26,1999,36,35,...,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>
6,ENG-Premier League,2526,Arsenal,Eberechi Eze,ENG,"MF,FW",27,1998,32,21,...,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>
7,ENG-Premier League,2526,Arsenal,Ethan Nwaneri,ENG,MF,18,2007,6,0,...,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>
8,ENG-Premier League,2526,Arsenal,Gabriel Jesus,BRA,FW,28,1997,14,3,...,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>
9,ENG-Premier League,2526,Arsenal,Gabriel Magalhães,BRA,DF,27,1997,32,30,...,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>


In [7]:
master_df.columns

Index(['league', 'season', 'team', 'player', 'nation', 'pos', 'age', 'born',
       'Playing Time_MP', 'Playing Time_Starts', 'Playing Time_Min',
       'Playing Time_90s', 'Performance_Gls', 'Performance_Ast',
       'Performance_G+A', 'Performance_G-PK', 'Performance_PK',
       'Performance_PKatt', 'Performance_CrdY', 'Performance_CrdR',
       'Per 90 Minutes_Gls', 'Per 90 Minutes_Ast', 'Per 90 Minutes_G+A',
       'Per 90 Minutes_G-PK', 'Per 90 Minutes_G+A-PK', '90s', 'Standard_Gls',
       'Standard_Sh', 'Standard_SoT', 'Standard_SoT%', 'Standard_Sh/90',
       'Standard_SoT/90', 'Standard_G/Sh', 'Standard_G/SoT', 'Standard_PK',
       'Standard_PKatt', 'Starts_Starts', 'Starts_Mn/Start', 'Starts_Compl',
       'Subs_Subs', 'Subs_Mn/Sub', 'Subs_unSub', 'Team Success_PPM',
       'Team Success_onG', 'Team Success_onGA', 'Team Success_+/-',
       'Team Success_+/-90', 'Team Success_On-Off', 'Performance_GA',
       'Performance_GA90', 'Performance_SoTA', 'Performance_Saves',
     

In [8]:
#master_df.to_csv("../../data/raw/fbref_top5leagues_players_2025.csv", index=False)

In [9]:
understat = sd.Understat(leagues=top_5_leagues, seasons=2025)
understat_df = understat.read_player_season_stats()

# 2. Move the MultiIndex into normal columns
understat_df = understat_df.reset_index()

# Look at the columns Understat provides
print(understat_df.columns)

[05/31/26 17:49:49] INFO     Saving cached data to C:\Users\vibha\soccerdata\data\Understat          _common.py:250

[2026-05-31 17:49:51] INFO     TLSLibrary:_load_library:397 - Successfully loaded TLS library: D:\Football Project\venv\Lib\site-packages\tls_requests\bin\tls-client-xgo-1.13.1-windows-amd64.dll


[05/31/26 17:49:51] INFO     Successfully loaded TLS library: D:\Football                          libraries.py:397
                             Project\venv\Lib\site-packages\tls_requests\bin\tls-client-xgo-1.13.1                 
                             -windows-amd64.dll                                                                    

Index(['league', 'season', 'team', 'player', 'league_id', 'season_id',
       'team_id', 'player_id', 'position', 'matches', 'minutes', 'goals', 'xg',
       'np_goals', 'np_xg', 'assists', 'xa', 'shots', 'key_passes',
       'yellow_cards', 'red_cards', 'xg_chain', 'xg_buildup'],
      dtype='str')


In [10]:
understat_df.head(20)

,league,season,team,player,league_id,season_id,team_id,player_id,position,matches,...,np_goals,np_xg,assists,xa,shots,key_passes,yellow_cards,red_cards,xg_chain,xg_buildup
0,ENG-Premier League,2526,Arsenal,Ben White,1,2025,83,7298,D S,12,...,0,0.282924,1,1.557781,5,8,0,0,4.476397,3.178621
1,ENG-Premier League,2526,Arsenal,Bukayo Saka,1,2025,83,7322,F M S,31,...,6,7.940027,5,8.552798,71,62,2,0,19.112561,8.037839
2,ENG-Premier League,2526,Arsenal,Christian Nørgaard,1,2025,83,7083,M S,7,...,0,0.068695,0,0.0,2,0,0,0,0.820646,0.771904
3,ENG-Premier League,2526,Arsenal,Cristhian Mosquera,1,2025,83,10024,D S,20,...,0,0.085854,0,0.293453,1,5,4,0,5.388046,5.184731
4,ENG-Premier League,2526,Arsenal,David Raya,1,2025,83,9676,GK,37,...,0,0.0,0,0.149394,0,3,1,0,6.412498,6.321507
5,ENG-Premier League,2526,Arsenal,Declan Rice,1,2025,83,5553,D M S,36,...,4,3.720961,5,8.15945,40,63,3,0,18.490908,12.956868
6,ENG-Premier League,2526,Arsenal,Eberechi Eze,1,2025,83,8706,F M S,33,...,7,4.807457,2,2.633517,63,17,1,0,10.355983,4.07407
7,ENG-Premier League,2526,Arsenal,Ethan Nwaneri,1,2025,83,11132,S,6,...,0,0.329211,0,0.048408,6,1,0,0,1.166223,0.984738
8,ENG-Premier League,2526,Arsenal,Gabriel,1,2025,83,5613,D S,32,...,3,4.649078,4,2.72503,27,7,4,0,8.894045,8.309923
9,ENG-Premier League,2526,Arsenal,Gabriel Jesus,1,2025,83,5543,F S,14,...,3,3.976481,0,0.208589,19,3,3,0,4.364025,1.269479


In [11]:
#understat_df.to_csv("../../data/raw/understat_top5leagues_players_2025.csv", index=False)

In [12]:
time=5

zero_minutes_count = (master_df['Playing Time_Min'] < time).sum()

print(f"Number of players with 0 minutes: {zero_minutes_count}")

zero_minutes_count = (understat_df['minutes'] < time).sum()

print(f"Number of players with 0 minutes: {zero_minutes_count}")


Number of players with 0 minutes: 54
Number of players with 0 minutes: 83


In [13]:
fbref_keys = master_df['player'] + " | " + master_df['team']
understat_keys = understat_df['player'] + " | " + understat_df['team']

exact_matches = fbref_keys.isin(understat_keys).sum()

print(f"Total rows in FBref: {len(master_df)}")
print(f"Total rows in Understat: {len(understat_df)}")
print(f"Perfect (Player + Team) Matches: {exact_matches}")

Total rows in FBref: 2839
Total rows in Understat: 2775
Perfect (Player + Team) Matches: 1776


In [14]:
fbref_teams = set(master_df['team'].dropna().unique())
understat_teams = set(understat_df['team'].dropna().unique())

matched_teams = fbref_teams.intersection(understat_teams)
fbref_only = fbref_teams.difference(understat_teams)      
understat_only = understat_teams.difference(fbref_teams)  

print("--- TEAM NAMING DIAGNOSTIC ---")
print(f"Total FBref Teams: {len(fbref_teams)}")
print(f"Total Understat Teams: {len(understat_teams)}")
print(f"Perfect Matches: {len(matched_teams)}\n")

print(f"--- FBREF NAMES ({len(fbref_only)} teams looking for a match) ---")
for team in sorted(fbref_only):
    print(f"- {team}")

print(f"\n--- UNDERSTAT NAMES ({len(understat_only)} teams looking for a match) ---")
for team in sorted(understat_only):
    print(f"- {team}")

--- TEAM NAMING DIAGNOSTIC ---
Total FBref Teams: 96
Total Understat Teams: 96
Perfect Matches: 76

--- FBREF NAMES (20 teams looking for a match) ---
- Alavés
- Atlético Madrid
- Dortmund
- Gladbach
- Heidenheim
- Hellas Verona
- Köln
- Leeds United
- Leverkusen
- Manchester Utd
- Milan
- Oviedo
- Paris Saint-Germain
- Parma
- RB Leipzig
- St Pauli
- Stuttgart
- Tottenham Hotspur
- West Ham United
- Wolves

--- UNDERSTAT NAMES (20 teams looking for a match) ---
- AC Milan
- Alaves
- Atletico Madrid
- Bayer Leverkusen
- Borussia Dortmund
- Borussia M.Gladbach
- FC Cologne
- FC Heidenheim
- Leeds
- Manchester United
- Paris Saint Germain
- Parma Calcio 1913
- RasenBallsport Leipzig
- Real Oviedo
- St. Pauli
- Tottenham
- Verona
- VfB Stuttgart
- West Ham
- Wolverhampton Wanderers


In [15]:
master_team_mapping = {
    "Alaves": "Alavés",
    "Atletico Madrid": "Atlético Madrid",
    "Borussia Dortmund": "Dortmund",
    "Borussia M.Gladbach": "Gladbach",
    "FC Heidenheim": "Heidenheim",
    "Verona": "Hellas Verona",
    "FC Cologne": "Köln",
    "Leeds": "Leeds United",
    "Bayer Leverkusen": "Leverkusen",
    "Manchester United": "Manchester Utd",
    "AC Milan": "Milan",
    "Real Oviedo": "Oviedo",
    "Paris Saint Germain": "Paris Saint-Germain",
    "Parma Calcio 1913": "Parma",
    "RasenBallsport Leipzig": "RB Leipzig",
    "St. Pauli": "St Pauli",
    "VfB Stuttgart": "Stuttgart",
    "Tottenham": "Tottenham Hotspur",
    "West Ham": "West Ham United",
    "Wolverhampton Wanderers": "Wolves"
}

temp_understat_teams = understat_df['team'].replace(master_team_mapping)

fbref_keys = master_df['player'] + " | " + master_df['team']
understat_keys = understat_df['player'] + " | " + temp_understat_teams

exact_matches = fbref_keys.isin(understat_keys).sum()

fbref_set = set(fbref_keys.dropna())
understat_set = set(understat_keys.dropna())

unmatched_fbref_players = fbref_set.difference(understat_set)

print("--- PLAYER NAME DISCREPANCY AUDIT ---")
print(f"Total rows in FBref: {len(master_df)}")
print(f"Total rows in Understat: {len(understat_df)}")
print(f"Perfect (Player + Team) Matches: {exact_matches}")
print(f"Total Failed Matches in FBref: {len(unmatched_fbref_players)}\n")

print("--- SAMPLE OF FAILED MATCHES (Look for Accents/Name Differences) ---")

for player_key in sorted(list(unmatched_fbref_players))[:20]:
    print(f"- {player_key}")

--- PLAYER NAME DISCREPANCY AUDIT ---
Total rows in FBref: 2839
Total rows in Understat: 2775
Perfect (Player + Team) Matches: 2245
Total Failed Matches in FBref: 594

--- SAMPLE OF FAILED MATCHES (Look for Accents/Name Differences) ---
- Aarón Escandell | Oviedo
- Abakar Sylla | Strasbourg
- Abde Ezzalzouli | Real Betis
- Abde Rebbach | Alavés
- Abdel Abqar | Getafe
- Abdelhamid Ait Boudlal | Rennes


- Abdoul Coulibaly | Werder Bremen
- Abdoul Ouattara | Strasbourg
- Abdoulaye Ndiaye | Parma
- Abdukodir Khusanov | Manchester City
- Abdulai Juma Bah | Nice
- Adam Boayar | Elche
- Adam Dźwigała | St Pauli
- Adam Hložek | Hoffenheim
- Adam Marušić | Lazio
- Adama Traoré | West Ham United
- Adrian Bernabe | Parma
- Adrian Šemper | Pisa
- Adrià Altimira | Villarreal
- Adrià Pedrosa | Sevilla


In [16]:

understat_metrics = ['player', 'team', 'xg', 'xa', 'shots', 'key_passes', 'xg_chain', 'xg_buildup']

pass_1_matches = pd.merge(
    master_df,
    understat_df[understat_metrics],
    on=['player', 'team'],
    how='inner'
)

print(f"Pass 1 Perfect Matches: {len(pass_1_matches)}")

Pass 1 Perfect Matches: 1776


In [17]:
fbref_leftovers = master_df[~master_df['player'].isin(pass_1_matches['player'])].copy()
understat_leftovers = understat_df[~understat_df['player'].isin(pass_1_matches['player'])].copy()

fbref_leftovers = fbref_leftovers.rename(columns={
    'Performance_Gls': 'goals', 
    'Performance_Ast': 'assists'
})

def get_initials(name):
    return "".join([word[0].upper() for word in str(name).split() if word])

fbref_leftovers['initials'] = fbref_leftovers['player'].apply(get_initials)
understat_leftovers['initials'] = understat_leftovers['player'].apply(get_initials)

heuristic_metrics = ['team', 'goals', 'assists', 'initials', 'xg', 'xa', 'shots', 'key_passes', 'xg_chain', 'xg_buildup']

pass_2_matches = pd.merge(
    fbref_leftovers,
    understat_leftovers[heuristic_metrics],
    on=['team', 'goals', 'assists', 'initials'],
    how='inner'
)

print(f"Pass 2 Heuristic Matches: {len(pass_2_matches)}")

if len(pass_2_matches) > 0:
    print("\nSample of Heuristic Matches:")
    print(pass_2_matches[['player', 'team', 'goals', 'assists', 'initials']].head(10))

Pass 2 Heuristic Matches: 240

Sample of Heuristic Matches:
                 player         team  goals  assists initials
0       Viktor Gyökeres      Arsenal     14        1       VG
1           Emi Buendía  Aston Villa      6        2       EB
2     Emiliano Martínez  Aston Villa      0        1       EM
3            Matty Cash  Aston Villa      3        3       MC
4        Bafodé Diakité  Bournemouth      0        0       BD
5   Hamed Junior Traorè  Bournemouth      0        1      HJT
6  Veljko Milosavljević  Bournemouth      0        0       VM
7     Caoimhín Kelleher    Brentford      0        0       CK
8       Yehor Yarmoliuk    Brentford      1        2       YY
9       Martin Dúbravka      Burnley      0        0       MD


In [18]:
import unicodedata
def normalize_text(series):
    """Converts to lowercase, strips spaces, and removes accents."""
    return series.astype(str).str.lower().str.strip().apply(
        lambda x: ''.join(c for c in unicodedata.normalize('NFD', x) if unicodedata.category(c) != 'Mn')
    )


fbref_clean = master_df.copy()
understat_clean = understat_df.copy()

understat_clean['team'] = understat_clean['team'].replace(master_team_mapping)

fbref_clean['match_player'] = normalize_text(fbref_clean['player'])
fbref_clean['match_team'] = normalize_text(fbref_clean['team'])

understat_clean['match_player'] = normalize_text(understat_clean['player'])
understat_clean['match_team'] = normalize_text(understat_clean['team'])

understat_pass1_metrics = ['match_player', 'match_team', 'player', 'xg', 'xa', 'shots', 'key_passes', 'xg_chain', 'xg_buildup']

pass_1_matches = pd.merge(
    fbref_clean,
    understat_clean[understat_pass1_metrics].rename(columns={'player': 'u_player'}), # Keep Understat name for tracking
    on=['match_player', 'match_team'],
    how='inner'
)

print(f"Pass 1 (Text) Perfect Matches: {len(pass_1_matches)}")


fbref_leftovers = fbref_clean[~fbref_clean['player'].isin(pass_1_matches['player'])].copy()
understat_leftovers = understat_clean[~understat_clean['player'].isin(pass_1_matches['u_player'])].copy()

fbref_leftovers = fbref_leftovers.rename(columns={'Performance_Gls': 'goals', 'Performance_Ast': 'assists'})

def get_initials(name):
    return "".join([word[0].upper() for word in str(name).split() if word])

fbref_leftovers['initials'] = fbref_leftovers['player'].apply(get_initials)
understat_leftovers['initials'] = understat_leftovers['player'].apply(get_initials)

heuristic_metrics = ['team', 'goals', 'assists', 'initials', 'xg', 'xa', 'shots', 'key_passes', 'xg_chain', 'xg_buildup']

pass_2_matches = pd.merge(
    fbref_leftovers,
    understat_leftovers[heuristic_metrics],
    on=['team', 'goals', 'assists', 'initials'],
    how='inner'
)

print(f"Pass 2 (Heuristic) Matches: {len(pass_2_matches)}")

pass_1_clean = pass_1_matches.drop(columns=['match_player', 'match_team', 'u_player'])
pass_2_clean = pass_2_matches.drop(columns=['goals', 'assists', 'initials'])

all_successful_matches = pd.concat([pass_1_clean, pass_2_clean], ignore_index=True)

understat_final_cols = ['player', 'team', 'xg', 'xa', 'shots', 'key_passes', 'xg_chain', 'xg_buildup']

ultimate_master = pd.merge(
    master_df,
    all_successful_matches[understat_final_cols],
    on=['player', 'team'],
    how='left'
)

total_players = len(ultimate_master)
matched_outfielders = ultimate_master[(~ultimate_master['pos'].str.contains("GK", na=False)) & (ultimate_master['xg'].notna())]
print(f"\nFINAL TOTAL MATCHED: {len(matched_outfielders)} out of {total_players} total play")

Pass 1 (Text) Perfect Matches: 2463
Pass 2 (Heuristic) Matches: 105

FINAL TOTAL MATCHED: 2387 out of 2842 total play


In [19]:
#ultimate_master.to_csv("../../data/raw/fbref_and_understat_top5leagues_players_2025.csv", index=False)